In [0]:
%run ./common/_utils_config

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.raw")
dbutils.fs.mkdirs(f"{raw_path}/prices")
dbutils.fs.mkdirs(f"{raw_path}/generation")

In [0]:
import requests
from datetime import date, timedelta

endpoints = {
    "prices": ("https://apidatos.ree.es/es/datos/mercados/precios-mercados-tiempo-real", "hour"),
    "generation": ("https://apidatos.ree.es/es/datos/generacion/estructura-generacion", "day"),
}

def download_chunk(name, start_day, end_day):
    url, time_trunc = endpoints[name]
    params = {"start_date": f"{start_day}T00:00", "end_date": f"{end_day}T23:59", "time_trunc": time_trunc}
    response = requests.get(url, params=params, timeout=60)
    if response.status_code != 200:
        print(f"{name} {start_day}->{end_day}: HTTP {response.status_code}")
        return
    dbutils.fs.put(f"{raw_path}/{name}/{name}_{start_day}_{end_day}.json", response.text, overwrite=True)
    print(f"{name} {start_day}->{end_day}: OK")

today = date.today()
chunk_start = today - timedelta(days=days_back)
while chunk_start <= today:
    chunk_end = min(chunk_start + timedelta(days=6), today)
    for name in endpoints:
        download_chunk(name, chunk_start, chunk_end)
    chunk_start = chunk_end + timedelta(days=1)

# Precios de mañana (REE los publica por la tarde; si aún no están, dará error y no pasa nada)
download_chunk("prices", today + timedelta(days=1), today + timedelta(days=1))

In [0]:
files = dbutils.fs.ls(f"{raw_path}/prices")
print(dbutils.fs.head(files[0].path, 1500))